### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [2]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content="<think>\nOkay, so I need to figure out why parrots talk. Let me start by recalling what I know about parrots. They're birds, right? And some species are known for mimicking human speech. I've seen parrots on TV that can say words and even phrases. But why do they do that? Is it just imitation, or is there a deeper reason?\n\nFirst, maybe I should think about their natural behavior. Do wild parrots talk? I think wild parrots use their voices for communication. They might have different calls for various situations, like warning others of predators or calling mates. So maybe talking in captivity is an extension of that natural behavior. But then, why do they specifically mimic human speech? Is it because they're in an environment with humans and they pick it up?\n\nAnother angle is their intelligence. Parrots are considered among the smartest birds. They have large brains relative to their body size. So maybe they can learn complex sounds, which includes human speech. 

In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston")
print(response)

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking about the weather in Boston. Let me check the tools available. There\'s a function called get_weather that takes a location parameter. Since the user mentioned Boston, I need to call that function with "Boston" as the location. I should make sure the parameters are correctly formatted in JSON. Let me structure the tool call accordingly.\n', 'tool_calls': [{'id': 'r5em5dt1z', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 96, 'prompt_tokens': 153, 'total_tokens': 249, 'completion_time': 0.149635667, 'completion_tokens_details': {'reasoning_tokens': 72}, 'prompt_time': 0.006789038, 'prompt_tokens_details': None, 'queue_time': 0.090861276, 'total_time': 0.156424705}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': N

In [6]:
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool: get_weather
Args: {'location': 'Boston'}


## Tool Execution Loop

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. Let me know if you need more details!


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there\'s no typo. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'pjw29vf7x', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 153, 'total_tokens': 246, 'completion_time': 0.149933359, 'completion_tokens_details': {'reasoning_tokens': 69}, 'prompt_time': 0.006235231, 'prompt_tokens_details': None, 'queue_time': 0.091225464, 'total_time': 0.15616859}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_r